# Preparación, limpieza y estructuración de los datos de propiedad horizontal en Medellín

**Evidencia:** GA1-220501094-01-AA1-EV07 – Preparación, limpieza y estructuración de datos del proyecto individual con Pandas

**Aprendiz:** Maicol Esneider Posada Callejas · **Ficha:** 3223873 · **Instructor:** Carlos Navia

**Pregunta analítica del proyecto:** ¿En qué comunas de Medellín se concentran los predios habitacionales en propiedad horizontal y qué proporción representan frente al total de predios habitacionales de cada comuna?

**Pregunta orientadora de la actividad:** ¿Cómo preparar y estructurar los datos del proyecto individual para obtener información confiable que permita responder la pregunta analítica planteada?

Este cuaderno aplica las técnicas de las lecciones 2, 3 y 4 del material *Preparar y estructurar datos con Python* a los cuatro datasets del proyecto. Parte de los problemas encontrados en el diagnóstico de calidad (`data_quality.ipynb` y `reports/data_quality.md`).

**Contenido**

0. Carga de los datos originales y estado inicial
1. Limpieza y estandarización (Lección 2)
2. Filtrado, ordenamiento, agrupación e indicadores (Lección 3)
3. Combinación y estructuración (Lección 4)
4. Comparación de la calidad antes y después

## 0. Carga de los datos originales y estado inicial

Los cuatro archivos están en `data/raw/` y no se modifican. Primero se cargan tal como vienen, para dejar registrado su estado inicial.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')
SALIDA.mkdir(exist_ok=True)

predios_crudo     = pd.read_csv(RUTA / 'informacion_predios.csv')
lotes_crudo       = pd.read_csv(RUTA / 'informacion_lotes.csv')
propiedades_crudo = pd.read_csv(RUTA / 'medellin_properties.csv')
inmuebles_crudo   = pd.read_csv(RUTA / 'houses_medellin.csv')

print('pandas', pd.__version__)

pandas 3.0.5


In [2]:
# Textos que el diagnóstico encontró como valores de relleno (datos faltantes disfrazados)
RELLENOS = ['--', 'INVALID', 'No definida', 'Otro', 'Más de 10']

def perfil(df, nombre):
    """Resume el estado de calidad de un DataFrame en una fila."""
    return {
        'dataset': nombre,
        'filas': len(df),
        'columnas': df.shape[1],
        'nulos': int(df.isna().sum().sum()),
        'filas_duplicadas': int(df.duplicated().sum()),
        'valores_relleno': int(df.isin(RELLENOS).sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    }

estado_inicial = pd.DataFrame([
    perfil(predios_crudo, 'predios'),
    perfil(lotes_crudo, 'lotes'),
    perfil(propiedades_crudo, 'propiedades_kaggle'),
    perfil(inmuebles_crudo, 'inmuebles_kaggle'),
])
estado_inicial

,dataset,filas,columnas,nulos,filas_duplicadas,valores_relleno,memoria_mb
0,predios,1041413,15,5800,0,590022,156.9
1,lotes,308129,8,0,0,0,22.8
2,propiedades_kaggle,9999,12,6028,100,127,1.4
3,inmuebles_kaggle,430,14,2365,0,128,0.1


Problemas del diagnóstico de calidad que se tratan en este cuaderno:

| Dataset | Problemas |
|---|---|
| Predios | `CBML` y `Matricula` leídos como números (se pierde el cero inicial), `NroPropietarios` siempre en 0, "--" en `Urbanizacion`, estrato 0, comunas mal escritas, códigos de `UsoPredial` sin descripción |
| Lotes | `CBML` como número, `ZonaGeo` como decimal, lotes con área 0 y con pisos o área construida incoherentes |
| Propiedades (Kaggle) | Duplicados, coordenadas como texto con "INVALID" y fuera de Medellín, "Más de 10" en garajes, áreas y precios imposibles, estrato 0 |
| Inmuebles (Kaggle) | Columna de índice sobrante, montos y áreas como texto, filas que solo tienen el precio, "No definida" y "Otro" como valores de relleno |

---
# 1. Limpieza y estandarización (Lección 2)

Para cada dataset se construye un pipeline: una cadena de funciones pequeñas unidas con `pipe()`. Cada función hace un solo paso de limpieza y la función `anotar` registra el estado de la tabla después de cada paso. Al final, las reglas de calidad se comprueban con `assert`.

## 1.1 Registro de cambios y funciones reutilizables

In [3]:
registro = []

def anotar(df, tabla, paso):
    """Guarda filas, nulos, duplicados y memoria de la tabla en el registro de cambios."""
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'columnas': df.shape[1],
        'nulos': int(df.isna().sum().sum()),
        'duplicados': int(df.duplicated().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df


def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)


def renombrar(df, nombres):
    return df.rename(columns=nombres)


def estandarizar_texto(df, columnas, correcciones=None):
    """Quita espacios sobrantes, corrige nombres mal escritos y deja Mayúscula Inicial."""
    df = df.copy()
    for col in columnas:
        texto = (df[col].astype('string')
                 .str.replace(r'\s+', ' ', regex=True)
                 .str.strip()
                 .str.upper())
        if correcciones:
            texto = texto.replace(correcciones)
        df[col] = texto.str.title()
    return df


def codigo_cbml(serie):
    """Convierte el CBML leído como número a texto de 11 dígitos, recuperando el cero inicial."""
    return serie.astype('int64').astype(str).str.zfill(11)


CORRECCIONES_COMUNAS = {'MANRRIQUE': 'MANRIQUE', 'LA AMERIA': 'LA AMÉRICA'}

`estandarizar_texto`, `quitar_duplicados`, `renombrar` y `codigo_cbml` se usan en varios datasets. Las columnas se renombran a un estándar único en minúsculas con guion bajo, porque el diagnóstico mostró que los nombres del CSV no coinciden con los del diccionario de MEData.

## 1.2 Información Predios

**Decisiones de limpieza:**

| Problema | Decisión | Justificación |
|---|---|---|
| `CBML` como número | Convertir a texto de 11 dígitos | Es un código: comuna (2) + barrio (2) + manzana (3) + lote (4) |
| `Matricula` como número | Convertir a texto | Es un identificador, no una cantidad |
| `NroPropietarios` siempre 0 | Eliminar la columna | No aporta información |
| "--" en `Urbanizacion` | Convertir a nulo | Es un valor de relleno |
| Estrato 0 | Convertir a nulo | El estrato solo se asigna a vivienda; 0 significa "sin estrato" |
| "MANRRIQUE", "LA AMERIA" | Corregir | Son errores de escritura |
| Valores nulos en `Urbanizacion` | Conservar | Solo es un nombre descriptivo; no se usa en el análisis |

In [4]:
NOMBRES_PREDIOS = {
    'codComuna': 'cod_comuna', 'codBarrio': 'cod_barrio', 'Manzana': 'manzana', 'CedCat': 'lote',
    'CBML': 'cbml', 'Matricula': 'matricula', 'PorDes': 'por_des', 'Estrato': 'estrato',
    'Urbanizacion': 'urbanizacion', 'UsoPredial': 'uso_predial', 'IndicadorU/R': 'zona',
    'NroPropietarios': 'nro_propietarios', 'nomComuna': 'comuna', 'DS_BARRIO': 'barrio',
    'AvaluoTotal': 'avaluo_total',
}

def tipos_predios(df):
    df = df.copy()
    df['cbml'] = codigo_cbml(df['cbml'])
    df['matricula'] = df['matricula'].astype(str)
    return df.astype({'cod_comuna': 'int8', 'cod_barrio': 'int8', 'manzana': 'int16', 'lote': 'int16',
                      'uso_predial': 'int8', 'zona': 'category'})

def quitar_columna_vacia(df):
    assert (df['nro_propietarios'] == 0).all()
    return df.drop(columns='nro_propietarios')

def rellenos_a_nulo_predios(df):
    df = df.copy()
    df['urbanizacion'] = df['urbanizacion'].replace('--', pd.NA)
    df['estrato'] = df['estrato'].replace(0, pd.NA).astype('Int8')
    return df

def clasificar_territorio(df):
    df = df.copy()
    df['tipo_territorio'] = np.where(df['cod_comuna'] <= 16, 'comuna', 'corregimiento')
    return df

predios = (predios_crudo
    .pipe(anotar, 'predios', '0 original')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'predios', '1 sin duplicados')
    .pipe(renombrar, NOMBRES_PREDIOS)
    .pipe(tipos_predios)
    .pipe(anotar, 'predios', '2 nombres y tipos')
    .pipe(quitar_columna_vacia)
    .pipe(rellenos_a_nulo_predios)
    .pipe(anotar, 'predios', '3 rellenos a nulo')
    .pipe(estandarizar_texto, ['comuna', 'barrio', 'urbanizacion'], CORRECCIONES_COMUNAS)
    .pipe(clasificar_territorio)
    .pipe(anotar, 'predios', '4 texto estandarizado'))

predios.head()

,cod_comuna,cod_barrio,manzana,lote,cbml,matricula,por_des,estrato,urbanizacion,uso_predial,zona,comuna,barrio,avaluo_total,tipo_territorio
0,11,5,55,1,11050550001,257792,0.085,<NA>,Torres De Rio No 1 Y 2,10,U,Laureles,Los Conquistadores,17500000.0,comuna
1,11,5,55,1,11050550001,257821,0.085,<NA>,Torres De Rio No 1 Y 2,10,U,Laureles,Los Conquistadores,17500000.0,comuna
2,11,9,28,27,11090280027,258113,33.823,5,<NA>,1,U,Laureles,Las Acacias,326925000.0,comuna
3,80,98,11,15,80980110015,1245992,0.020,<NA>,Reservas Del Prado Etapa 2,10,U,San Antonio De Prado,Zona Expansion San Antonio,6342000.0,corregimiento
4,80,98,11,15,80980110015,1246000,0.020,<NA>,Reservas Del Prado Etapa 2,10,U,San Antonio De Prado,Zona Expansion San Antonio,6342000.0,corregimiento


In [5]:
print(predios.dtypes)
print('\nComunas y corregimientos:', sorted(predios['comuna'].unique().tolist()))

cod_comuna             int8
cod_barrio             int8
manzana               int16
lote                  int16
cbml                    str
matricula               str
por_des             float64
estrato                Int8
urbanizacion         string
uso_predial            int8
zona               category
comuna               string
barrio               string
avaluo_total        float64
tipo_territorio         str
dtype: object

Comunas y corregimientos: ['Altavista', 'Aranjuez', 'Belen', 'Buenos Aires', 'Castilla', 'Doce De Octubre', 'El Poblado', 'Guayabal', 'La América', 'La Candelaria', 'Laureles', 'Manrique', 'Palmitas', 'Popular', 'Robledo', 'San Antonio De Prado', 'San Cristobal', 'San Javier', 'Santa Cruz', 'Santa Elena', 'Villa Hermosa']


### Uso predial: ¿qué código corresponde a vivienda?

El diagnóstico encontró que `UsoPredial` viene en códigos sin tabla de equivalencias. Como el estrato socioeconómico en Colombia solo se asigna a la vivienda, se cruza el código de uso con la existencia de estrato:

In [6]:
uso_vs_estrato = (predios
    .assign(tiene_estrato=predios['estrato'].notna())
    .groupby('uso_predial')
    .agg(predios=('matricula', 'size'),
         con_estrato=('tiene_estrato', 'sum'),
         por_des_mediana=('por_des', 'median')))
uso_vs_estrato['pct_con_estrato'] = (uso_vs_estrato['con_estrato'] / uso_vs_estrato['predios'] * 100).round(2)
uso_vs_estrato.sort_values('predios', ascending=False)

,predios,con_estrato,por_des_mediana,pct_con_estrato
uso_predial,,,,
1,645912,645907,24.4765,100.00
10,240381,2158,0.0700,0.90
2,99326,1386,0.4750,1.40
4,21022,249,100.0000,1.18
9,16788,94,100.0000,0.56
3,6989,581,100.0000,8.31
5,6983,3,100.0000,0.04
8,3702,2,100.0000,0.05
7,307,9,100.0000,2.93


**Resultado:** el código **1** tiene 645.912 predios y **645.907 tienen estrato (100 %)**. En los demás códigos, como mucho el 8,3 % de los predios tiene estrato. Además, los códigos 10 y 2 tienen porcentajes de copropiedad muy pequeños (mediana de 0,07 y 0,48), típicos de parqueaderos, cuartos útiles y locales dentro de edificios.

**Decisión:** se toma `uso_predial = 1` como **vivienda**. Es una inferencia sustentada en los datos, no una tabla oficial; si Catastro entrega la tabla de códigos, este criterio se debe confirmar.

In [7]:
predios['es_vivienda'] = predios['uso_predial'] == 1
print('Predios de vivienda:', predios['es_vivienda'].sum())

Predios de vivienda: 645912


## 1.3 Información Lotes

| Problema | Decisión | Justificación |
|---|---|---|
| `CBML` como número | Convertir a texto de 11 dígitos | Es la clave para unir con Predios |
| `ZonaGeo` como decimal | Convertir a texto sin el ".0" | Es un código |
| Área del lote igual a 0 | Convertir a nulo | Un lote no puede medir 0 m² |
| Área construida 0 con pisos, o área construida con 0 pisos | Marcar con una columna `pisos_incoherentes` | No se sabe cuál de los dos datos está mal; se conservan y se excluyen de los cálculos de pisos |
| Nombres de comuna mal escritos | Corregir | Igual que en Predios |

In [8]:
NOMBRES_LOTES = {'CBML': 'cbml', 'codComuna': 'cod_comuna', 'nomComuna': 'comuna', 'AreaLote': 'area_lote',
                 'AreaCons': 'area_construida', 'Zona': 'zona_codigo', 'ZonaGeo': 'zona_geo',
                 'numPisos': 'num_pisos'}

def tipos_lotes(df):
    df = df.copy()
    df['cbml'] = codigo_cbml(df['cbml'])
    df['zona_geo'] = df['zona_geo'].astype('Int64').astype('string')
    return df.astype({'cod_comuna': 'int8', 'zona_codigo': 'int16', 'num_pisos': 'int16'})

def marcar_inconsistencias_lotes(df):
    df = df.copy()
    df['area_lote'] = df['area_lote'].replace(0, np.nan)
    df['pisos_incoherentes'] = (((df['area_construida'] == 0) & (df['num_pisos'] > 0)) |
                                ((df['area_construida'] > 0) & (df['num_pisos'] == 0)))
    return df

lotes = (lotes_crudo
    .pipe(anotar, 'lotes', '0 original')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'lotes', '1 sin duplicados')
    .pipe(renombrar, NOMBRES_LOTES)
    .pipe(tipos_lotes)
    .pipe(anotar, 'lotes', '2 nombres y tipos')
    .pipe(marcar_inconsistencias_lotes)
    .pipe(anotar, 'lotes', '3 inconsistencias marcadas')
    .pipe(estandarizar_texto, ['comuna'], CORRECCIONES_COMUNAS)
    .pipe(anotar, 'lotes', '4 texto estandarizado'))

print('Lotes con área 0 convertida a nulo:', lotes['area_lote'].isna().sum())
print('Lotes con pisos incoherentes:', lotes['pisos_incoherentes'].sum())
print('Longitud de zona_geo:')
print(lotes['zona_geo'].str.len().value_counts())
lotes.head()

Lotes con área 0 convertida a nulo: 80
Lotes con pisos incoherentes: 1111
Longitud de zona_geo:
zona_geo
15    306559
3       1566
8          2
7          1
13         1
Name: count, dtype: Int64


,cbml,cod_comuna,comuna,area_lote,area_construida,zona_codigo,zona_geo,num_pisos,pisos_incoherentes
0,01010020003,1,Popular,33.0,85.0,440,122121422212440,2,False
1,01010020010,1,Popular,145.0,188.0,440,122121422212440,2,False
2,01010020014,1,Popular,55.0,173.0,440,203121422313440,3,False
3,01010030013,1,Popular,123.0,102.0,440,111131422212440,2,False
4,01010030021,1,Popular,51.0,129.0,440,111131422212440,2,False


`zona_geo` quedó como texto: 306.559 lotes tienen el código completo de 15 dígitos y 1.566 tienen un código de 3 dígitos (igual al formato de `zona_codigo`). Esa diferencia se conserva y se documenta, porque `zona_geo` no se usa para responder la pregunta.

## 1.4 Medellín Properties 2023 (Kaggle)

| Problema | Decisión |
|---|---|
| 100 filas duplicadas, más las que aparecen al estandarizar | Eliminar antes y después de limpiar |
| Coordenadas como texto con "INVALID" | Convertir a número; "INVALID" queda como nulo |
| Coordenadas fuera de Medellín | Convertir a nulo (el inmueble se conserva) |
| "Más de 10" en garajes | Convertir a nulo y marcar en `garajes_mas_de_10` |
| Área 0, estrato 0 y barrio "Otro" | Convertir a nulo |
| `age` es un código de rango | Convertir a categoría ordenada |
| Precios y áreas imposibles | Marcar en `valor_atipico`; se excluyen de los cálculos |

In [9]:
NOMBRES_PROPIEDADES = {'neighbourhood': 'barrio', 'latitude': 'latitud', 'longitude': 'longitud',
                       'property_type': 'tipo', 'price': 'precio', 'rooms': 'habitaciones', 'baths': 'banos',
                       'area': 'area', 'administration_price': 'administracion', 'age': 'antiguedad_codigo',
                       'garages': 'garajes', 'stratum': 'estrato'}

def coordenadas(df):
    df = df.copy()
    df['latitud'] = pd.to_numeric(df['latitud'], errors='coerce')
    df['longitud'] = pd.to_numeric(df['longitud'], errors='coerce')
    dentro = df['latitud'].between(6.1, 6.4) & df['longitud'].between(-75.75, -75.45)
    df.loc[~dentro, ['latitud', 'longitud']] = np.nan
    return df

def corregir_valores_propiedades(df):
    df = df.copy()
    df['garajes_mas_de_10'] = df['garajes'] == 'Más de 10'
    df['garajes'] = pd.to_numeric(df['garajes'], errors='coerce').astype('Int8')
    df['area'] = df['area'].replace(0, np.nan)
    df['estrato'] = df['estrato'].replace(0, pd.NA).astype('Int8')
    df['habitaciones'] = df['habitaciones'].astype('Int8')
    df['banos'] = df['banos'].astype('Int8')
    df['antiguedad_codigo'] = pd.Categorical(df['antiguedad_codigo'].astype('Int8'),
                                             categories=[0, 1, 2, 3, 4], ordered=True)
    df['valor_atipico'] = (~df['precio'].between(5e7, 1e11)) | (df['area'] > 2000)
    return df

def barrio_desconocido(df):
    df = df.copy()
    df['barrio'] = df['barrio'].replace('Otro', pd.NA)
    return df

propiedades = (propiedades_crudo
    .pipe(anotar, 'propiedades_kaggle', '0 original')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'propiedades_kaggle', '1 sin duplicados')
    .pipe(renombrar, NOMBRES_PROPIEDADES)
    .pipe(coordenadas)
    .pipe(anotar, 'propiedades_kaggle', '2 coordenadas')
    .pipe(corregir_valores_propiedades)
    .pipe(estandarizar_texto, ['barrio', 'tipo'])
    .pipe(barrio_desconocido)
    .pipe(anotar, 'propiedades_kaggle', '3 valores y texto')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'propiedades_kaggle', '4 duplicados que aparecieron al limpiar'))

print('Coordenadas válidas:', propiedades['latitud'].notna().sum(), 'de', len(propiedades))
print('Inmuebles con valor atípico:', propiedades['valor_atipico'].sum())
print(propiedades.dtypes)

Coordenadas válidas: 9727 de 9883
Inmuebles con valor atípico: 35
barrio                 string
latitud               float64
longitud              float64
tipo                   string
precio                  int64
habitaciones             Int8
banos                    Int8
area                  float64
administracion        float64
antiguedad_codigo    category
garajes                  Int8
estrato                  Int8
garajes_mas_de_10        bool
valor_atipico            bool
dtype: object


## 1.5 Precios inmuebles Medellín (Kaggle)

| Problema | Decisión |
|---|---|
| Columna `Unnamed: 0` (índice guardado por error) | Eliminar |
| Filas que solo tienen el precio | Eliminar: no aportan nada para relacionar la administración con el estrato o el área |
| Montos con `$`, `COP`, `*m²`, puntos de miles y comas decimales | Convertir a número |
| Áreas como texto con espacios y comas | Convertir a número; área privada 0 a nulo |
| "No definida" y "Otro" | Convertir a nulo |
| Antigüedad en rangos de texto | Convertir a categoría ordenada |
| Duplicados que aparecen después de limpiar | Eliminar |

In [10]:
NOMBRES_INMUEBLES = {'Habitaciones': 'habitaciones', 'Baños': 'banos', 'Estrato': 'estrato',
                     'Antigüedad': 'antiguedad', 'Piso N°': 'piso', 'Administración': 'administracion',
                     'Precio m²': 'precio_m2', 'Parqueaderos': 'parqueaderos', 'Estado': 'estado',
                     'Tipo de apartamento': 'tipo_apartamento', 'Precio': 'precio',
                     'Área construída (m²)': 'area_construida', 'Área privada (m²)': 'area_privada'}

def texto_a_numero(serie):
    """'$ 4.032.258,06*m²' -> 4032258.06 ; 'No definida' -> nulo"""
    limpio = (serie.astype('string')
              .str.replace(r'[^0-9,]', '', regex=True)
              .str.replace(',', '.', regex=False)
              .replace('', pd.NA))
    return pd.to_numeric(limpio, errors='coerce')

def quitar_indice_y_filas_vacias(df):
    df = df.drop(columns='Unnamed: 0')
    sin_precio = df.drop(columns='Precio')
    return df[sin_precio.notna().any(axis=1)].reset_index(drop=True)

def convertir_inmuebles(df):
    df = df.copy()
    for col in ['administracion', 'precio_m2', 'area_construida', 'area_privada']:
        df[col] = texto_a_numero(df[col])
    df['area_privada'] = df['area_privada'].replace(0, np.nan)
    df['piso'] = pd.to_numeric(df['piso'], errors='coerce').astype('Int8')
    for col in ['habitaciones', 'banos', 'estrato', 'parqueaderos']:
        df[col] = df[col].astype('Int8')
    df['antiguedad'] = pd.Categorical(df['antiguedad'],
        categories=['menor a 1 año', '1 a 8 años', '9 a 15 años', '16 a 30 años', 'más de 30 años'],
        ordered=True)
    df['valor_atipico'] = ~df['precio'].between(5e7, 1e11)
    return df

inmuebles = (inmuebles_crudo
    .pipe(anotar, 'inmuebles_kaggle', '0 original')
    .pipe(quitar_indice_y_filas_vacias)
    .pipe(anotar, 'inmuebles_kaggle', '1 sin índice ni filas vacías')
    .pipe(renombrar, NOMBRES_INMUEBLES)
    .pipe(convertir_inmuebles)
    .pipe(anotar, 'inmuebles_kaggle', '2 montos y áreas a número')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'inmuebles_kaggle', '3 sin duplicados'))

print(inmuebles.dtypes)
inmuebles.head()

habitaciones            Int8
banos                   Int8
estrato                 Int8
antiguedad          category
piso                    Int8
administracion         Int64
precio_m2            Float64
parqueaderos            Int8
estado                   str
tipo_apartamento         str
precio                 int64
area_construida      Float64
area_privada         Float64
valor_atipico           bool
dtype: object


,habitaciones,banos,estrato,antiguedad,piso,administracion,precio_m2,parqueaderos,estado,tipo_apartamento,precio,area_construida,area_privada,valor_atipico
0,1,2,4,1 a 8 años,3,<NA>,4032258.06,<NA>,NaN,NaN,250000000,62.0,62.0,False
1,1,2,6,9 a 15 años,<NA>,<NA>,6428571.43,2,NaN,NaN,675000000,105.0,<NA>,False
2,1,1,5,NaN,3,167000,4700000.0,2,NaN,NaN,235000000,50.0,<NA>,False
3,1,1,4,NaN,<NA>,<NA>,6187500.0,1,Excelente,NaN,198000000,32.0,<NA>,False
4,2,2,6,menor a 1 año,<NA>,180000,7421875.0,1,Excelente,NaN,475000000,64.0,64.0,False


## 1.6 Registro de cambios

In [11]:
registro_cambios = pd.DataFrame(registro)
registro_cambios.to_csv(SALIDA / 'cleaning_log.csv', index=False)
registro_cambios

,tabla,paso,filas,columnas,nulos,duplicados,memoria_mb
0,predios,0 original,1041413,15,5800,0,156.9
1,predios,1 sin duplicados,1041413,15,5800,0,156.9
2,predios,2 nombres y tipos,1041413,15,5800,0,132.5
3,predios,3 rellenos a nulo,1041413,14,986846,0,116.7
4,predios,4 texto estandarizado,1041413,15,986846,0,132.3
5,lotes,0 original,308129,8,0,0,22.8
6,lotes,1 sin duplicados,308129,8,0,0,22.8
7,lotes,2 nombres y tipos,308129,8,0,0,24.9
8,lotes,3 inconsistencias marcadas,308129,9,80,0,25.2
9,lotes,4 texto estandarizado,308129,9,80,0,25.3


**Interpretación del registro de cambios:**

- **Predios:** no se perdió ninguna fila. Los nulos suben de 5.800 a 986.846 porque los valores de relleno quedaron como nulos reales: 590.022 "--" en `urbanizacion` y 391.024 estratos en 0. Antes esos datos faltantes estaban escondidos; ahora pandas los reconoce. La memoria baja de 156,9 a 133,3 MB con tipos más pequeños.
- **Lotes:** no se perdió ninguna fila. Aparecen 80 nulos: los lotes con área 0.
- **Medellín Properties:** se eliminaron 100 duplicados exactos y otros 16 que aparecieron al limpiar coordenadas y texto (de 9.999 a 9.883 filas).
- **Precios inmuebles:** se eliminaron 121 filas que solo tenían el precio y 98 duplicados (de 430 a 211 filas). El dataset queda pequeño, pero confiable.

El registro se guardó en `data/processed/cleaning_log.csv`.

## 1.7 Validación con `assert`

In [12]:
def validar_predios(df):
    assert df['matricula'].is_unique, 'matrícula repetida'
    assert (df['cbml'].str.len() == 11).all(), 'CBML sin 11 dígitos'
    assert df['cbml'].str.isdigit().all(), 'CBML con caracteres no numéricos'
    assert df['estrato'].dropna().between(1, 6).all(), 'estrato fuera de 1 a 6'
    assert df['por_des'].between(0, 100).all(), 'porcentaje de copropiedad fuera de 0 a 100'
    assert not df['comuna'].isin(['Manrrique', 'La Ameria']).any(), 'comunas mal escritas'
    assert (df['urbanizacion'] != '--').all(), 'quedan valores de relleno'
    return 'Predios: validación superada'

def validar_lotes(df):
    assert df['cbml'].is_unique, 'CBML repetido'
    assert (df['cbml'].str.len() == 11).all(), 'CBML sin 11 dígitos'
    assert (df['area_lote'].dropna() > 0).all(), 'áreas en 0'
    return 'Lotes: validación superada'

def validar_propiedades(df):
    assert df.duplicated().sum() == 0, 'duplicados'
    assert df['latitud'].dropna().between(6.1, 6.4).all(), 'latitud fuera de Medellín'
    assert df['estrato'].dropna().between(1, 6).all(), 'estrato fuera de 1 a 6'
    assert pd.api.types.is_numeric_dtype(df['garajes']), 'garajes no numérico'
    return 'Propiedades: validación superada'

def validar_inmuebles(df):
    assert 'Unnamed: 0' not in df.columns, 'columna de índice'
    assert df.duplicated().sum() == 0, 'duplicados'
    for col in ['administracion', 'precio_m2', 'area_construida']:
        assert pd.api.types.is_numeric_dtype(df[col]), f'{col} no numérico'
    return 'Inmuebles: validación superada'

print(validar_predios(predios))
print(validar_lotes(lotes))
print(validar_propiedades(propiedades))
print(validar_inmuebles(inmuebles))

Predios: validación superada


Lotes: validación superada


Propiedades: validación superada
Inmuebles: validación superada


Los cuatro datasets cumplen todas las reglas de calidad definidas. Si alguna fallara, `assert` detendría el cuaderno con el mensaje del problema.

---
# 2. Filtrado, ordenamiento, agrupación e indicadores (Lección 3)

La pregunta analítica se traduce a "métrica por grupo": **número y porcentaje de viviendas en propiedad horizontal por comuna**.

**Criterio de propiedad horizontal.** Un predio está en propiedad horizontal cuando comparte el lote con otros predios. En Catastro esto se ve en el porcentaje de copropiedad (`por_des`): una casa que ocupa todo su lote tiene 100, y un apartamento tiene una fracción. Por eso, **vivienda en propiedad horizontal = predio de vivienda con `por_des` menor que 100**.

## 2.1 Filtros

In [13]:
# Condición booleana: solo vivienda
viviendas = predios[predios['es_vivienda']].copy()
print('Viviendas en toda la ciudad:', len(viviendas))

# isin(): solo las 16 comunas urbanas (la pregunta es por comunas)
viviendas_comunas = viviendas[viviendas['cod_comuna'].isin(range(1, 17))].copy()
print('Viviendas en las 16 comunas:', len(viviendas_comunas))
print('Viviendas en los 5 corregimientos:', len(viviendas) - len(viviendas_comunas))

Viviendas en toda la ciudad: 645912
Viviendas en las 16 comunas: 575614
Viviendas en los 5 corregimientos: 70298


In [14]:
# Indicador de propiedad horizontal
viviendas_comunas['en_ph'] = viviendas_comunas['por_des'] < 100
print('Viviendas en propiedad horizontal:', viviendas_comunas['en_ph'].sum())
print('Viviendas que ocupan todo su lote:', (~viviendas_comunas['en_ph']).sum())
print('Viviendas en PH con porcentaje registrado en 0:', (viviendas_comunas['por_des'] == 0).sum())

# between(): viviendas en PH de estratos 1 a 3
ph = viviendas_comunas[viviendas_comunas['en_ph']]
print('Viviendas en PH de estratos 1 a 3:', ph['estrato'].between(1, 3).sum())

# query(): viviendas en PH de estratos 5 y 6 en El Poblado
print('Viviendas en PH estratos 5 y 6 en El Poblado:',
      len(ph.query("estrato >= 5 and comuna == 'El Poblado'")))

Viviendas en propiedad horizontal: 432859
Viviendas que ocupan todo su lote: 142755
Viviendas en PH con porcentaje registrado en 0: 9725


Viviendas en PH de estratos 1 a 3: 253935


Viviendas en PH estratos 5 y 6 en El Poblado: 46886


## 2.2 Tamaño de cada propiedad horizontal

No es lo mismo una casa dividida en dos apartamentos que un conjunto de 300 apartamentos. Domux, el software del proyecto, sirve sobre todo para conjuntos y edificios con administración. Por eso se cuenta cuántas viviendas en PH hay en cada lote con `transform` y se clasifica el tamaño con `pd.cut`.

In [15]:
viviendas_comunas['viviendas_ph_lote'] = (viviendas_comunas
    .groupby('cbml')['en_ph'].transform('sum'))

viviendas_comunas['en_ph_grande'] = viviendas_comunas['en_ph'] & (viviendas_comunas['viviendas_ph_lote'] > 20)

ph = viviendas_comunas[viviendas_comunas['en_ph']].copy()
ph['tamano_ph'] = pd.cut(ph['viviendas_ph_lote'],
                         bins=[0, 4, 20, np.inf],
                         labels=['pequeña (1 a 4 viviendas)', 'mediana (5 a 20)', 'grande (más de 20)'])
ph['tamano_ph'].value_counts().sort_index()

tamano_ph
pequeña (1 a 4 viviendas)    180192
mediana (5 a 20)              78067
grande (más de 20)           174600
Name: count, dtype: int64

**Interpretación:** de las 432.859 viviendas en propiedad horizontal de las 16 comunas, el **41,6 %** está en lotes pequeños (1 a 4 viviendas, casas divididas o edificios de pocos pisos), el **18,0 %** en lotes medianos (5 a 20) y el **40,3 %** (174.600 viviendas) en lotes grandes de más de 20 viviendas, que son los conjuntos y edificios con administración.

## 2.3 Indicadores por comuna con `groupby()` y `agg()`

| Indicador | Definición |
|---|---|
| **I1.** Viviendas en propiedad horizontal | Número de viviendas con `por_des` menor que 100 |
| **I2.** Proporción de PH en la comuna | I1 dividido por el total de viviendas de la comuna × 100 |
| **I3.** Participación en la ciudad | I1 de la comuna dividido por el total de viviendas en PH de las 16 comunas × 100 |
| **I4.** Viviendas en PH grande | Viviendas en lotes con más de 20 viviendas en PH (conjuntos y edificios) |
| **I5.** Proporción de PH grande | I4 dividido por I1 × 100 |

In [16]:
indicadores = (viviendas_comunas
    .groupby(['cod_comuna', 'comuna'])
    .agg(viviendas=('matricula', 'size'),
         viviendas_ph=('en_ph', 'sum'),
         viviendas_ph_grande=('en_ph_grande', 'sum'))
    .reset_index())

indicadores['pct_ph_en_comuna'] = (indicadores['viviendas_ph'] / indicadores['viviendas'] * 100).round(1)
indicadores['pct_ph_ciudad'] = (indicadores['viviendas_ph'] / indicadores['viviendas_ph'].sum() * 100).round(1)
indicadores['pct_ph_grande'] = (indicadores['viviendas_ph_grande'] / indicadores['viviendas_ph'] * 100).round(1)

indicadores.sort_values('viviendas_ph', ascending=False).reset_index(drop=True)

,cod_comuna,comuna,viviendas,viviendas_ph,viviendas_ph_grande,pct_ph_en_comuna,pct_ph_ciudad,pct_ph_grande
0,16,Belen,65730,55701,30400,84.7,12.9,54.6
1,14,El Poblado,49835,48402,42474,97.1,11.2,87.8
2,7,Robledo,50457,39003,20633,77.3,9.0,52.9
3,11,Laureles,41369,37389,15150,90.4,8.6,40.5
4,9,Buenos Aires,43174,33565,14994,77.7,7.8,44.7
5,12,La América,33292,29418,12974,88.4,6.8,44.1
6,13,San Javier,38159,25753,9166,67.5,5.9,35.6
7,10,La Candelaria,28425,25103,16300,88.3,5.8,64.9
8,6,Doce De Octubre,35919,24628,2648,68.6,5.7,10.8
9,5,Castilla,29261,23325,1786,79.7,5.4,7.7


### Ordenamiento: las comunas que más concentran propiedad horizontal

In [17]:
print('Top 5 por número de viviendas en PH (I1):')
print(indicadores.nlargest(5, 'viviendas_ph')[['comuna', 'viviendas_ph', 'pct_ph_ciudad']].to_string(index=False))

print('\nTop 5 por proporción de PH dentro de la comuna (I2):')
print(indicadores.nlargest(5, 'pct_ph_en_comuna')[['comuna', 'pct_ph_en_comuna']].to_string(index=False))

print('\nTop 5 por viviendas en PH grande (I4):')
print(indicadores.nlargest(5, 'viviendas_ph_grande')[['comuna', 'viviendas_ph_grande', 'pct_ph_grande']].to_string(index=False))

print('\nComunas con menor proporción de PH (I2):')
print(indicadores.nsmallest(3, 'pct_ph_en_comuna')[['comuna', 'pct_ph_en_comuna']].to_string(index=False))

Top 5 por número de viviendas en PH (I1):
      comuna  viviendas_ph  pct_ph_ciudad
       Belen         55701           12.9
  El Poblado         48402           11.2
     Robledo         39003            9.0
    Laureles         37389            8.6
Buenos Aires         33565            7.8

Top 5 por proporción de PH dentro de la comuna (I2):
       comuna  pct_ph_en_comuna
   El Poblado              97.1
     Laureles              90.4
   La América              88.4
La Candelaria              88.3
        Belen              84.7

Top 5 por viviendas en PH grande (I4):
       comuna  viviendas_ph_grande  pct_ph_grande
   El Poblado                42474           87.8
        Belen                30400           54.6
      Robledo                20633           52.9
La Candelaria                16300           64.9
     Laureles                15150           40.5

Comunas con menor proporción de PH (I2):
       comuna  pct_ph_en_comuna
      Popular              39.0
Villa Hermosa 

**Interpretación de los indicadores:**

- **I1 y I3 (cantidad y participación):** el 75,2 % de las viviendas de las 16 comunas (432.859 de 575.614) está en propiedad horizontal. **Belén** es la comuna con más viviendas en PH (55.701, el 12,9 % de la ciudad), seguida de **El Poblado** (48.402), **Robledo** (39.003), **Laureles** (37.389) y **Buenos Aires** (33.565). Esas cinco comunas reúnen el 49,5 % de la propiedad horizontal de la ciudad.
- **I2 (proporción dentro de la comuna):** en **El Poblado** el 97,1 % de las viviendas está en PH; también superan el 88 % Laureles, La América y La Candelaria. Las comunas con menor proporción son **Popular** (39,0 %), Villa Hermosa (52,3 %) y Manrique (53,7 %), donde predominan las casas que ocupan todo su lote.
- **I4 e I5 (propiedad horizontal grande):** El Poblado tiene 42.474 viviendas en conjuntos o edificios de más de 20 viviendas (87,8 % de su PH), seguido de Belén (30.400), Robledo (20.633), La Candelaria (16.300) y Laureles (15.150). Estas cinco comunas concentran el 71,6 % de las viviendas en PH grande. En Popular y Manrique, menos del 4 % de la PH es grande.

## 2.4 Tablas dinámicas

In [18]:
ph_comuna_estrato = ph.pivot_table(index='comuna', columns='estrato', values='matricula',
                                   aggfunc='count', fill_value=0)
ph_comuna_estrato

estrato,1,2,3,4,5,6
comuna,,,,,,
Aranjuez,1414,5715,14718,20,0,0
Belen,335,3945,20356,17777,13171,117
Buenos Aires,769,3875,22450,5731,319,421
Castilla,279,2543,19946,555,2,0
Doce De Octubre,2798,12619,9209,2,0,0
El Poblado,6,117,226,1167,9132,37754
Guayabal,70,1390,6999,3780,18,0
La América,44,200,7006,13753,8286,129
La Candelaria,2,885,7750,14568,1895,3


In [19]:
ph_comuna_tamano = (pd.crosstab(ph['comuna'], ph['tamano_ph'], normalize='index')
                      .mul(100).round(1)
                      .sort_values('grande (más de 20)', ascending=False))
ph_comuna_tamano

tamano_ph,pequeña (1 a 4 viviendas),mediana (5 a 20),grande (más de 20)
comuna,,,
El Poblado,1.5,10.7,87.8
La Candelaria,17.6,17.5,64.9
Belen,29.8,15.6,54.6
Robledo,39.2,7.9,52.9
Buenos Aires,43.4,11.9,44.7
La América,26.8,29.1,44.1
Laureles,15.9,43.6,40.5
San Javier,45.8,18.7,35.6
Villa Hermosa,63.4,18.0,18.6


**Interpretación de las tablas dinámicas:**

- **Comuna × estrato:** la propiedad horizontal de El Poblado es casi toda de estratos 5 y 6 (46.886 viviendas); la de Laureles, de estratos 4 y 5; y la de Belén está repartida entre los estratos 3, 4 y 5. En Popular y Santa Cruz la PH es casi solo de estratos 1 y 2, y en Manrique tres de cada cuatro viviendas en PH son de esos estratos.
- **Comuna × tamaño:** en El Poblado el 87,8 % de la PH es grande y en La Candelaria el 64,9 %. En Popular, Manrique, Santa Cruz y Doce de Octubre, entre el 78 % y el 83 % es pequeña: son casas divididas en varias viviendas, no conjuntos con administración.

## 2.5 Indicador complementario: valor de la administración (Kaggle)

La administración es el cobro mensual que existe en la propiedad horizontal. Con los apartamentos en venta de *Medellín Properties 2023*, sin valores atípicos y con administración mayor que 0, se calcula la mediana por estrato.

In [20]:
apartamentos = propiedades[(propiedades['tipo'] == 'Apartamento') &
                           (~propiedades['valor_atipico']) &
                           (propiedades['administracion'] > 0)]

administracion_estrato = (apartamentos
    .groupby('estrato')
    .agg(apartamentos=('precio', 'size'),
         administracion_mediana=('administracion', 'median'),
         area_mediana=('area', 'median')))
administracion_estrato

,apartamentos,administracion_mediana,area_mediana
estrato,,,
1,8,187650.0,55.5
2,110,106000.0,55.0
3,1230,166500.0,60.0
4,2145,267000.0,74.0
5,1203,370000.0,100.0
6,482,623000.0,140.0


**Interpretación:** en los apartamentos en venta, la mediana de la administración mensual sube con el estrato: 166.500 pesos en estrato 3, 267.000 en estrato 4, 370.000 en estrato 5 y 623.000 en estrato 6. Los estratos 1 y 2 tienen muy pocos apartamentos en la muestra (8 y 110), así que sus valores no son confiables. Este indicador complementa la pregunta: en las comunas con PH grande y de estratos altos, la administración es más alta, lo que indica conjuntos con más servicios que gestionar.

## 2.6 Interpretación y relación con la pregunta analítica

**Respuesta a la pregunta analítica con los datos de 2021:**

1. **¿Dónde se concentran las viviendas en propiedad horizontal?** En número, en **Belén, El Poblado, Robledo, Laureles y Buenos Aires**, que reúnen casi la mitad de la PH de la ciudad.
2. **¿Qué proporción representan dentro de cada comuna?** Van del **39,0 % en Popular** al **97,1 % en El Poblado**. En promedio, 3 de cada 4 viviendas de las comunas de Medellín están en propiedad horizontal.
3. **Matiz importante para el proyecto:** no toda la propiedad horizontal es igual. En las comunas populares es sobre todo pequeña (casas divididas), mientras que en El Poblado, Belén, Robledo, La Candelaria y Laureles hay más conjuntos y edificios grandes. Esas comunas son las más indicadas para empezar con Domux, porque allí hay más copropiedades con administración que necesitan un sistema de gestión.

---
# 3. Combinación y estructuración (Lección 4)

## 3.1 Relaciones entre los datasets

| Datasets | Relación | ¿Se combinan? |
|---|---|---|
| Predios y Lotes | Clave `cbml`: muchos predios por cada lote (relación muchos a uno) | **Sí.** Así cada vivienda recibe el área y los pisos de su lote |
| Predios/Lotes y Medellín Properties | No hay clave común: Kaggle tiene nombres de barrio escritos de forma libre y coordenadas, no `cbml` | **No.** Unir por nombre de barrio sería una combinación artificial y poco confiable |
| Predios/Lotes y Precios inmuebles | El dataset no tiene ninguna columna de ubicación | **No.** No hay forma válida de relacionarlos |
| Medellín Properties y Precios inmuebles | Columnas distintas, sin clave común | **No.** Se dejan como datasets de apoyo independientes |

### Verificación de la clave antes de unir

In [21]:
print('CBML único en lotes:', lotes['cbml'].is_unique)
print('Viviendas cuyo lote existe en lotes:', viviendas_comunas['cbml'].isin(lotes['cbml']).mean() * 100, '%')

auditoria = viviendas_comunas[['cbml']].merge(lotes[['cbml']], on='cbml', how='left', indicator=True)
auditoria['_merge'].value_counts()

CBML único en lotes: True


Viviendas cuyo lote existe en lotes: 100.0 %


_merge
both          575614
left_only          0
right_only         0
Name: count, dtype: int64

## 3.2 Tabla consolidada de viviendas con `merge()`

In [22]:
filas_antes = len(viviendas_comunas)

viviendas_lote = viviendas_comunas.merge(
    lotes[['cbml', 'area_lote', 'area_construida', 'num_pisos', 'zona_geo', 'pisos_incoherentes']],
    on='cbml', how='left', validate='m:1')

assert len(viviendas_lote) == filas_antes, 'la unión cambió el número de filas'
assert viviendas_lote['matricula'].is_unique, 'matrícula repetida después de unir'
print(viviendas_lote.shape)
viviendas_lote.head()

(575614, 24)


,cod_comuna,cod_barrio,manzana,lote,cbml,matricula,por_des,estrato,urbanizacion,uso_predial,...,tipo_territorio,es_vivienda,en_ph,viviendas_ph_lote,en_ph_grande,area_lote,area_construida,num_pisos,zona_geo,pisos_incoherentes
0,11,9,28,27,11090280027,258113,33.823,5,<NA>,1,...,comuna,True,True,3,False,247.0,562.0,3,108212120451138,False
1,8,1,5,11,08010050011,258145,100.000,3,<NA>,1,...,comuna,True,False,0,False,116.0,209.0,3,111121461213485,False
2,14,16,3,1,14160030001,1247513,8.510,3,<NA>,1,...,comuna,True,True,4,False,196.0,338.0,3,108112300461081,False
3,13,8,14,20,13080140020,1247821,13.300,3,<NA>,1,...,comuna,True,True,5,False,230.0,510.0,3,111120461214494,False
4,9,7,19,28,09070190028,1280240,27.660,3,<NA>,1,...,comuna,True,True,3,False,216.0,419.0,3,111121451313494,False


La unión conserva exactamente una fila por vivienda: `validate='m:1'` confirma que cada lote aparece una sola vez en `lotes`, y el `assert` comprueba que no se crearon ni se perdieron filas.

## 3.3 Tabla estructurada de propiedades horizontales (un lote por fila)

In [23]:
# Los pisos de los lotes marcados como incoherentes no son confiables: se dejan como nulos
viviendas_lote['num_pisos_valido'] = viviendas_lote['num_pisos'].where(~viviendas_lote['pisos_incoherentes'])

lotes_ph = (viviendas_lote[viviendas_lote['en_ph']]
    .groupby('cbml')
    .agg(cod_comuna=('cod_comuna', 'first'),
         comuna=('comuna', 'first'),
         barrio=('barrio', 'first'),
         urbanizacion=('urbanizacion', 'first'),
         viviendas_ph=('matricula', 'size'),
         estrato=('estrato', 'median'),
         num_pisos=('num_pisos_valido', 'first'),
         area_lote=('area_lote', 'first'),
         avaluo_total=('avaluo_total', 'sum'))
    .reset_index())

lotes_ph['tamano_ph'] = pd.cut(lotes_ph['viviendas_ph'], bins=[0, 4, 20, np.inf],
                               labels=['pequeña (1 a 4 viviendas)', 'mediana (5 a 20)', 'grande (más de 20)'])

assert lotes_ph['cbml'].is_unique
assert lotes_ph['viviendas_ph'].sum() == viviendas_lote['en_ph'].sum()
print(lotes_ph.shape)
lotes_ph.nlargest(5, 'viviendas_ph')[['comuna', 'barrio', 'urbanizacion', 'viviendas_ph', 'num_pisos', 'estrato']]

(83005, 11)


,comuna,barrio,urbanizacion,viviendas_ph,num_pisos,estrato
46866,Buenos Aires,Barrio De Jesus,Bosque Verde,1179,NaN,3.0
46867,Buenos Aires,Barrio De Jesus,Senderos Bosque Verde,840,19.0,3.0
63991,La América,Santa Teresita,<NA>,830,15.0,5.0
54035,La Candelaria,Barrio Colon,Urbanizacion Torres De San Sebastian,780,21.0,3.0
78954,Belen,La Mota,Calamari,703,NaN,4.0


**Interpretación:** hay **83.005 lotes** en propiedad horizontal en las 16 comunas. El más grande es la urbanización Bosque Verde (Buenos Aires), con 1.179 viviendas. En los lotes marcados como incoherentes, el número de pisos queda vacío en lugar de mostrar un 0 falso.

## 3.4 Tabla final por comuna

Se unen con `merge()` los indicadores de viviendas y un resumen de los lotes en propiedad horizontal de cada comuna. Ambas tablas tienen una fila por comuna, así que la relación es uno a uno (`validate='1:1'`).

In [24]:
resumen_lotes = (lotes_ph
    .groupby('cod_comuna')
    .agg(lotes_ph=('cbml', 'size'),
         lotes_ph_grandes=('tamano_ph', lambda s: (s == 'grande (más de 20)').sum()),
         pisos_mediana=('num_pisos', 'median'))
    .reset_index())

resumen_comunas = indicadores.merge(resumen_lotes, on='cod_comuna', how='left', validate='1:1')

assert len(resumen_comunas) == 16
assert resumen_comunas['viviendas_ph'].sum() == viviendas_lote['en_ph'].sum()
assert resumen_comunas['pct_ph_en_comuna'].between(0, 100).all()
resumen_comunas.sort_values('viviendas_ph', ascending=False).reset_index(drop=True)

,cod_comuna,comuna,viviendas,viviendas_ph,viviendas_ph_grande,pct_ph_en_comuna,pct_ph_ciudad,pct_ph_grande,lotes_ph,lotes_ph_grandes,pisos_mediana
0,16,Belen,65730,55701,30400,84.7,12.9,54.6,7733,250,3.0
1,14,El Poblado,49835,48402,42474,97.1,11.2,87.8,1391,669,9.0
2,7,Robledo,50457,39003,20633,77.3,9.0,52.9,6767,263,3.0
3,11,Laureles,41369,37389,15150,90.4,8.6,40.5,4408,286,4.0
4,9,Buenos Aires,43174,33565,14994,77.7,7.8,44.7,6578,125,3.0
5,12,La América,33292,29418,12974,88.4,6.8,44.1,4224,167,3.0
6,13,San Javier,38159,25753,9166,67.5,5.9,35.6,5562,154,3.0
7,10,La Candelaria,28425,25103,16300,88.3,5.8,64.9,2435,186,3.0
8,6,Doce De Octubre,35919,24628,2648,68.6,5.7,10.8,7904,18,3.0
9,5,Castilla,29261,23325,1786,79.7,5.4,7.7,6817,29,3.0


### Corregimientos con `concat()`

La pregunta es por comunas, pero para que el dataset final esté completo se calcula lo mismo para los 5 corregimientos y se apilan las dos tablas con `concat()`. Tienen las mismas columnas, así que no es una combinación artificial.

In [25]:
viviendas_corr = viviendas[~viviendas['cod_comuna'].isin(range(1, 17))].copy()
viviendas_corr['en_ph'] = viviendas_corr['por_des'] < 100

resumen_corregimientos = (viviendas_corr
    .groupby(['cod_comuna', 'comuna'])
    .agg(viviendas=('matricula', 'size'), viviendas_ph=('en_ph', 'sum'))
    .reset_index())
resumen_corregimientos['pct_ph_en_comuna'] = (resumen_corregimientos['viviendas_ph'] /
                                              resumen_corregimientos['viviendas'] * 100).round(1)

territorios = pd.concat([
    resumen_comunas[['cod_comuna', 'comuna', 'viviendas', 'viviendas_ph', 'pct_ph_en_comuna']].assign(tipo='comuna'),
    resumen_corregimientos.assign(tipo='corregimiento'),
], ignore_index=True)

assert len(territorios) == 21
assert territorios['viviendas'].sum() == len(viviendas)
territorios.tail(6)

,cod_comuna,comuna,viviendas,viviendas_ph,pct_ph_en_comuna,tipo
15,16,Belen,65730,55701,84.7,comuna
16,50,Palmitas,1118,322,28.8,corregimiento
17,60,San Cristobal,29014,23674,81.6,corregimiento
18,70,Altavista,9752,6827,70.0,corregimiento
19,80,San Antonio De Prado,24889,18980,76.3,corregimiento
20,90,Santa Elena,5525,2251,40.7,corregimiento


## 3.5 Integridad y consistencia del resultado

In [26]:
comprobaciones = {
    'Cada vivienda aparece una sola vez': viviendas_lote['matricula'].is_unique,
    'Todas las viviendas tienen su lote': viviendas_lote['area_construida'].notna().all(),
    'Las viviendas en PH suman lo mismo en las tres tablas':
        viviendas_lote['en_ph'].sum() == lotes_ph['viviendas_ph'].sum() == resumen_comunas['viviendas_ph'].sum(),
    'Las 21 divisiones suman todas las viviendas': territorios['viviendas'].sum() == predios['es_vivienda'].sum(),
    'Los porcentajes están entre 0 y 100': territorios['pct_ph_en_comuna'].between(0, 100).all(),
    'Sin claves nulas': viviendas_lote['cbml'].notna().all() and lotes_ph['cbml'].notna().all(),
}
for regla, cumple in comprobaciones.items():
    print('OK   ' if cumple else 'FALLA', regla)
assert all(comprobaciones.values())

OK    Cada vivienda aparece una sola vez
OK    Todas las viviendas tienen su lote
OK    Las viviendas en PH suman lo mismo en las tres tablas
OK    Las 21 divisiones suman todas las viviendas
OK    Los porcentajes están entre 0 y 100
OK    Sin claves nulas


## 3.6 Exportación a `data/processed/`

In [27]:
predios.to_parquet(SALIDA / 'properties_clean.parquet', index=False)
lotes.to_parquet(SALIDA / 'lots_clean.parquet', index=False)
propiedades.to_csv(SALIDA / 'medellin_properties_clean.csv', index=False)
inmuebles.to_csv(SALIDA / 'houses_medellin_clean.csv', index=False)
viviendas_lote.to_parquet(SALIDA / 'residential_properties.parquet', index=False)
lotes_ph.to_csv(SALIDA / 'horizontal_property_lots.csv', index=False)
resumen_comunas.to_csv(SALIDA / 'comuna_summary.csv', index=False)
territorios.to_csv(SALIDA / 'territory_summary.csv', index=False)

for archivo in sorted(SALIDA.glob('*.*')):
    print(f'{archivo.name:35} {archivo.stat().st_size / 1e6:6.2f} MB')

cleaning_log.csv                      0.00 MB
comuna_summary.csv                    0.00 MB
horizontal_property_lots.csv          7.62 MB
houses_medellin_clean.csv             0.01 MB
lots_clean.parquet                    3.45 MB
medellin_properties_clean.csv         0.92 MB
properties_clean.parquet             21.52 MB
residential_properties.parquet       16.12 MB
territory_summary.csv                 0.00 MB


In [28]:
# Comprobación: los archivos guardados se leen igual que las tablas en memoria
assert pd.read_parquet(SALIDA / 'residential_properties.parquet').shape == viviendas_lote.shape
assert pd.read_csv(SALIDA / 'comuna_summary.csv').shape == resumen_comunas.shape
assert pd.read_parquet(SALIDA / 'properties_clean.parquet')['cbml'].str.len().eq(11).all()
print('Archivos exportados y verificados')

Archivos exportados y verificados


**Archivos generados en `data/processed/`:**

| Archivo | Contenido | Uso |
|---|---|---|
| `cleaning_log.csv` | Registro de cambios de los cuatro pipelines | Evidencia de la limpieza |
| `properties_clean.parquet` | Los 1.041.413 predios limpios | Dataset limpio de Predios |
| `lots_clean.parquet` | Los 308.129 lotes limpios | Dataset limpio de Lotes |
| `medellin_properties_clean.csv` | Inmuebles de Kaggle limpios (9.883) | Dataset de apoyo |
| `houses_medellin_clean.csv` | Inmuebles de Kaggle limpios (211) | Dataset de apoyo |
| `residential_properties.parquet` | Las 575.614 viviendas de las 16 comunas, unidas con su lote | Tabla consolidada para el análisis |
| `horizontal_property_lots.csv` | Un lote en propiedad horizontal por fila (83.005) | Tabla estructurada de copropiedades |
| `comuna_summary.csv` | Indicadores por comuna (16 filas) | Dataset final para responder la pregunta |
| `territory_summary.csv` | Viviendas y PH de las 16 comunas y los 5 corregimientos | Vista completa de la ciudad |

Se usa Parquet para las tablas grandes porque conserva los tipos de dato (por ejemplo, el `cbml` como texto con su cero inicial) y ocupa menos espacio; requiere `pyarrow`.

---
# 4. Comparación de la calidad antes y después

In [29]:
estado_final = pd.DataFrame([
    perfil(predios, 'predios'),
    perfil(lotes, 'lotes'),
    perfil(propiedades, 'propiedades_kaggle'),
    perfil(inmuebles, 'inmuebles_kaggle'),
])

comparacion = estado_inicial.merge(estado_final, on='dataset', suffixes=('_antes', '_despues'))
comparacion[['dataset', 'filas_antes', 'filas_despues',
             'filas_duplicadas_antes', 'filas_duplicadas_despues',
             'valores_relleno_antes', 'valores_relleno_despues',
             'nulos_antes', 'nulos_despues',
             'memoria_mb_antes', 'memoria_mb_despues']]

,dataset,filas_antes,filas_despues,filas_duplicadas_antes,filas_duplicadas_despues,valores_relleno_antes,valores_relleno_despues,nulos_antes,nulos_despues,memoria_mb_antes,memoria_mb_despues
0,predios,1041413,1041413,0,0,590022,0,5800,986846,156.9,133.3
1,lotes,308129,308129,0,0,0,0,0,80,22.8,25.3
2,propiedades_kaggle,9999,9883,100,0,127,0,6028,6298,1.4,0.9
3,inmuebles_kaggle,430,211,0,0,128,0,2365,599,0.1,0.0


**Interpretación de la comparación:**

- **Duplicados:** de 100 a 0 en Medellín Properties. Los otros datasets no tenían duplicados al inicio, salvo los que aparecieron en Precios inmuebles al quitar el índice.
- **Valores de relleno:** de 590.022 a 0 en Predios, de 127 a 0 en Medellín Properties y de 128 a 0 en Precios inmuebles. Ya no quedan datos faltantes disfrazados de texto.
- **Nulos:** en Predios suben, y es lo correcto: los valores de relleno y el estrato 0 ahora son nulos reales que pandas reconoce y excluye de los cálculos. En Precios inmuebles bajan de 2.365 a 599 porque se eliminaron las filas vacías.
- **Tipos:** los códigos (`cbml`, `matricula`, `zona_geo`) quedaron como texto, y los montos y áreas de Kaggle como números. Antes no se podían usar para cálculos.
- **Memoria:** baja en tres de los cuatro datasets. En Lotes sube un poco (de 22,8 a 25,3 MB) porque el `cbml` como texto ocupa más que como número, pero ahora es correcto.

---
# Conclusiones

1. **Los datos quedaron limpios, validados y listos para continuar.** Cuatro pipelines con `pipe()` corrigieron tipos, eliminaron duplicados, convirtieron valores de relleno en nulos y estandarizaron el texto. Las reglas de calidad se comprueban con `assert`, y el registro de cambios deja evidencia de cada paso.
2. **La pregunta analítica ya se puede responder.** El cruce del uso predial con el estrato permitió identificar la vivienda (código 1), y el porcentaje de copropiedad permitió identificar la propiedad horizontal. Belén, El Poblado, Robledo, Laureles y Buenos Aires concentran la mitad de las viviendas en PH, y la proporción va del 39 % en Popular al 97 % en El Poblado.
3. **La combinación se hizo solo donde existe una relación real.** Predios y Lotes se unieron por `cbml` con `validate='m:1'`, sin perder ni duplicar filas. Los datasets de Kaggle no se unieron porque no tienen una clave común con los de MEData; se limpiaron y quedaron como apoyo.
4. **Limitaciones que siguen vigentes:** los datos son de 2021, el código de vivienda es una inferencia sin tabla oficial, y la propiedad horizontal se estima con el porcentaje de copropiedad. El paso siguiente es visualizar los indicadores por comuna y validar el criterio con información de Catastro.